In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/Colab Notebooks/졸업프로젝트/OMRiDA')

# Preprocess

In [ ]:
# # bmp -> png 변환

# from pathlib import Path
# from PIL import Image

# def preprocess(image_path, output_path, ext="png"):
#     image_path = Path(image_path)
#     output_path = Path(output_path)
#     output_path.mkdir(parents=True, exist_ok=True)

#     for i, img_path in enumerate(sorted(image_path.glob(f"*.{ext}"))):
#         try:
#             with Image.open(img_path) as img:

#                 img = img.convert("RGB")
#                 img_name = img_path.name.split(".")[0]
#                 img_name = img_name + '.png'
#                 img.save(output_path / img_name, format="PNG")

#         except Exception as e:
#             print(f"❌ {img_path.name}: {e}")

# preprocess("data/preprocessed/test/hme/crohme/2014/hme_img", "data/preprocessed/test/hme/crohme/2014/hme_img_preprocessed", ext="bmp")
# preprocess("data/preprocessed/test/hme/crohme/2016/hme_img", "data/preprocessed/test/hme/crohme/2016/hme_img_preprocessed", ext="bmp")
# preprocess("data/preprocessed/test/hme/crohme/2019/hme_img", "data/preprocessed/test/hme/crohme/2019/hme_img_preprocessed", ext="bmp")

In [3]:
from collections import Counter
import numpy as np
from pathlib import Path
from PIL import Image

def analyze_image_stats(image_dir, image_ext="png", sample_limit=1000000):
    image_dir = Path(image_dir)
    sizes = []
    means = []
    stds = []
    names = []

    for i, img_path in enumerate(sorted(image_dir.glob(f"*.{image_ext}"))):
        if i >= sample_limit:
            break
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")  # (R, G, B) 보장
                sizes.append(img.size)    # (width, height)
                names.append(img_path.name)

                arr = np.array(img, dtype=np.float32) / 255.0  # (H, W, 3)
                means.append(np.mean(arr, axis=(0, 1)))  # (3,)
                stds.append(np.std(arr, axis=(0, 1)))    # (3,)
        except Exception as e:
            print(f"❌ {img_path.name}: {e}")

    means = np.stack(means, axis=0)  # (N, 3)
    stds = np.stack(stds, axis=0)    # (N, 3)
    channel_mean = np.mean(means, axis=0)  # (3,)
    channel_std = np.mean(stds, axis=0)    # (3,)

    widths, heights = zip(*sizes)
    print(len(sizes))
    print(f"Max size: {(int(np.max(widths)), int(np.max(heights)))}")
    print(f"Min size: {(int(np.min(widths)), int(np.min(heights)))}")
    print(f"Mean size: {(float(np.mean(widths)), float(np.mean(heights)))}")
    print(f"Median size: {(float(np.median(widths)), float(np.median(heights)))}")
    print(f"Top size: {Counter(sizes).most_common(5)}")
    print(f"Channel-wise Mean (R, G, B): {channel_mean.tolist()}")
    print(f"Channel-wise Std  (R, G, B): {channel_std.tolist()}")
    print()


if __name__ == "__main__":
    datasets = [
        {
            "name": "CROHME HME train",
            "path": "data/preprocessed/train/paired/crohme/hme_preprocessed",
            "image_ext":"png"
        },
        {
            "name": "CROHME PME train",
            "path": "data/preprocessed/train/paired/crohme/pme_preprocessed",
            "image_ext":"png"
        },
        {
            "name": "IM2LATEX HME train",
            "path": "data/preprocessed/train/paired/im2latex/hme",
            "image_ext":"png"
        },
        {
            "name": "IM2LATEX PME train",
            "path": "data/preprocessed/train/paired/im2latex/pme_cropped",
            "image_ext":"png"
        },
        {
            "name": "IM2LATEX PME test",
            "path": "data/preprocessed/test/pme/im2latex/img",
            "image_ext":"png"
        },
        {
            "name": "CROHME 2014 PME test",
            "path": "data/preprocessed/test/pme/crohme/2014/pme_img",
            "image_ext":"png"
        },
        {
            "name": "CROHME 2016 PME test",
            "path": "data/preprocessed/test/pme/crohme/2016/pme_img",
            "image_ext":"png"
        },
        {
            "name": "CROHME 2019 PME test",
            "path": "data/preprocessed/test/pme/crohme/2019/pme_img",
            "image_ext":"png"
        },
        {
            "name": "CROHME 2014 HME test",
            "path": "data/preprocessed/test/hme/crohme/2014/hme_img_preprocessed",
            "image_ext":"png"
        },
        {
            "name": "CROHME 2016 HME test",
            "path": "data/preprocessed/test/hme/crohme/2016/hme_img_preprocessed",
            "image_ext":"png"
        },
        {
            "name": "CROHME 2019 HME test",
            "path": "data/preprocessed/test/hme/crohme/2019/hme_img_preprocessed",
            "image_ext":"png"
        }
    ]

    for ds in datasets:
        print(f"📂 분석 중: {ds['name']}")
        analyze_image_stats(
            image_dir=ds["path"], image_ext=ds["image_ext"]
        )

📂 분석 중: CROHME HME train
8834
Max size: (2116, 481)
Min size: (26, 54)
Mean size: (314.3594068372198, 103.77484718134481)
Median size: (268.0, 93.0)
Top size: [((43, 54), 23), ((39, 54), 17), ((47, 54), 16), ((46, 54), 14), ((48, 54), 14)]
Channel-wise Mean (R, G, B): [0.07699312269687653, 0.07699312269687653, 0.07699312269687653]
Channel-wise Std  (R, G, B): [0.2632405757904053, 0.2632405757904053, 0.2632405757904053]

📂 분석 중: CROHME PME train
8834
Max size: (872, 125)
Min size: (8, 14)
Mean size: (149.29137423590672, 32.81333484265338)
Median size: (128.0, 28.0)
Top size: [((14, 19), 129), ((36, 19), 127), ((172, 33), 92), ((50, 28), 74), ((22, 19), 68)]
Channel-wise Mean (R, G, B): [0.8126205801963806, 0.8126205801963806, 0.8126205801963806]
Channel-wise Std  (R, G, B): [0.3815617859363556, 0.3815617859363556, 0.3815617859363556]

📂 분석 중: IM2LATEX HME train
3153
Max size: (801, 160)
Min size: (120, 31)
Mean size: (323.0196638122423, 59.759276879162705)
Median size: (360.0, 50.0)
Top

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


40
Max size: (1654, 2339)
Min size: (1654, 2339)
Mean size: (1654.0, 2339.0)
Median size: (1654.0, 2339.0)
Top size: [((1654, 2339), 40)]
Channel-wise Mean (R, G, B): [0.9992501139640808, 0.9992501139640808, 0.9992501139640808]
Channel-wise Std  (R, G, B): [0.025526434183120728, 0.025526434183120728, 0.025526434183120728]

📂 분석 중: CROHME 2014 PME test
986
Max size: (631, 97)
Min size: (14, 14)
Mean size: (156.59837728194725, 31.49898580121704)
Median size: (128.0, 28.0)
Top size: [((53, 28), 22), ((67, 25), 12), ((39, 19), 12), ((28, 19), 10), ((64, 25), 10)]
Channel-wise Mean (R, G, B): [0.8098785281181335, 0.8098785281181335, 0.8098785281181335]
Channel-wise Std  (R, G, B): [0.3852023780345917, 0.3852023780345917, 0.3852023780345917]

📂 분석 중: CROHME 2016 PME test
1147
Max size: (686, 75)
Min size: (17, 14)
Mean size: (166.512641673932, 31.345248474280734)
Median size: (142.0, 31.0)
Top size: [((61, 22), 12), ((67, 25), 11), ((17, 33), 10), ((67, 22), 9), ((144, 28), 8)]
Channel-wise 

# Custom Dataset

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
import torch


class FormulaDataset(Dataset):
    def __init__(self, image_dir, caption_path, transform=None, image_ext="png", vocab=None):
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.image_ext = image_ext
        self.vocab = vocab

        self.samples = []
        with open(caption_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) != 2:
                    continue
                file_id, latex = parts

                if '.' not in file_id:
                    filename = f"{file_id}.{self.image_ext}"
                else:
                    filename = file_id

                image_path = self.image_dir / filename
                if image_path.exists():
                    self.samples.append((image_path, latex))

    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        image_path, latex = self.samples[idx]
        image = Image.open(image_path).convert('L')

        if self.transform:
            image = self.transform(image)

        formula_ids = self.vocab.encode(latex)

        # ⚠️ 최소 길이 2 이상만 허용 (sos + token 또는 token + eos)
        if len(formula_ids) < 2:
            # 다음 인덱스로 순환 접근
            return self.__getitem__((idx + 1) % len(self.samples))

        formula_tensor = torch.tensor(formula_ids)

        return {
            "image": image,
            "formula": formula_tensor
        }

class PairedFormulaDataset(Dataset):
    def __init__(self, hme_dir, pme_dir, caption_path, transform_hme, transform_pme, image_exts=["png", "png"], vocab=None):
        self.hme_dir = Path(hme_dir)
        self.pme_dir = Path(pme_dir)
        self.transform_hme = transform_hme
        self.transform_pme = transform_pme
        self.vocab = vocab
        self.img_exts = image_exts

        self.samples = []
        with open(caption_path, "r") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) != 2:
                    continue
                img_name, formula = parts
                self.samples.append((img_name, formula))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, formula = self.samples[idx]

        if '.' not in img_name:
            img_hme_path = self.hme_dir / f"{img_name}.{self.img_exts[0]}"
            img_pme_path = self.pme_dir / f"{img_name}.{self.img_exts[1]}"

        else:
            img_hme_path = self.hme_dir / img_name
            img_pme_path = self.pme_dir / img_name

        img_hme = Image.open(img_hme_path).convert('L')
        img_pme = Image.open(img_pme_path).convert('L')

        if self.transform_hme:
            img_hme = self.transform_hme(img_hme)
        if self.transform_pme:
            img_pme = self.transform_pme(img_pme)

        token_ids = self.vocab.encode(formula)
        formula_tensor = torch.tensor(token_ids, dtype=torch.long)

        return {
            "img_hme": img_hme,
            "img_pme": img_pme,
            "formula": formula_tensor
        }

# Data utils

## 최소한의 성능 확인 위해 sub-optimal하게 구현

- 추후 고려해 볼 내용 (ablation study)
  - transform 함수를 미니배치단위로 동적으로 사이즈 맞추게끔해서 메모리 절약
  - 학습 시 정규화를 densenet pretrain이 아닌, 현재 데이터에 맞춰서 정규화 진행
  - densenet pretrain weight 사용 o,x 경우 비교
  - densenet pretrain 처음 채널을 1채널 (gray)로 할지 말지 유무에 따른 차이 비교
  - scale augmentation 이용

In [ ]:
from torchvision import transforms
import torch
from PIL import ImageOps

# class DynamicPadToSize:
#     def __init__(self, target_size=(2339, 1654), fill=255):
#         self.target_h, self.target_w = target_size
#         self.fill = fill

#     def __call__(self, img):
#         w, h = img.size
#         pad_left = max((self.target_w - w) // 2, 0)
#         pad_top = max((self.target_h - h) // 2, 0)
#         pad_right = max(self.target_w - w - pad_left, 0)
#         pad_bottom = max(self.target_h - h - pad_top, 0)

#         # 🔧 fill 값 조정: RGB면 튜플로 변환
#         fill_value = self.fill
#         if img.mode == "RGB" and isinstance(self.fill, int):
#             fill_value = (self.fill,) * 3

#         if pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0:
#             return ImageOps.expand(img, border=(pad_left, pad_top, pad_right, pad_bottom), fill=fill_value)
#         else:
#             return img

# ablation 실험 - 일반 resize대신 최대 크기에 맞춰 padding을 이용한 사이즈 통일, 단, 이 경우 늘어난 사이즈에 따라 연산량 매우 많아질 수 있을듯
# def get_pad_formula_transform(image_type: str, transform_config: dict):
#     cfg = transform_config.get(image_type)
#     if cfg is None:
#         raise ValueError(f"Unsupported image_type: {image_type}")

#     target_size = tuple(cfg["target_size"])
#     mean = cfg["mean"]
#     std = cfg["std"]

#     return transforms.Compose([
#         DynamicPadToSize(target_size=target_size, fill=255),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=mean, std=std)
#     ])

def get_formula_transform(target_size=(128,768)):

    return transforms.Compose([
        transforms.Resize(target_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])


def formula_collate_fn(batch, pad_idx=0):
    images = torch.stack([item["image"] for item in batch], dim=0)
    formulas = torch.nn.utils.rnn.pad_sequence(
        [item["formula"] for item in batch],
        batch_first=True,
        padding_value=pad_idx
    )
    return {"image": images, "formula": formulas}


def paired_collate_fn(batch, pad_idx=0):
    imgs_hme = torch.stack([item["img_hme"] for item in batch], dim=0)
    imgs_pme = torch.stack([item["img_pme"] for item in batch], dim=0)
    formulas = torch.nn.utils.rnn.pad_sequence(
        [item["formula"] for item in batch],
        batch_first=True,
        padding_value=pad_idx
    )
    return {
        "img_hme": imgs_hme,
        "img_pme": imgs_pme,
        "formula": formulas
    }

## Vocab

In [ ]:
from pathlib import Path
from typing import List
import re

def tokenize_formula(formula):
    ans = []
    tokens = formula.strip().split()

    for tok in tokens:
        chk = 0
        for sym in ["cm", "mm", "pt", "in", "ex", "em"]:
            if (tok[-2:] == sym) and tok[-3].isdigit():
                num, _ = tok.split(sym)
                num = ' '.join(num).split()
                ans.extend(num)
                ans.append(sym)
                chk = 1
                break

        if chk == 0:
            ans.append(tok)
    return ans


class Vocab:
    PAD_TOKEN = "<pad>"
    SOS_TOKEN = "<sos>"
    EOS_TOKEN = "<eos>"

    def __init__(self):
        self.tokens = [self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN]
        self.token2idx = {tok: idx for idx, tok in enumerate(self.tokens)}
        self.idx2token = self.tokens.copy()

    def build_vocab(self, formula_list: List[str]):
        for formula in formula_list:
            tokenized_formula = tokenize_formula(formula)
            for tok in tokenized_formula:
                if tok not in self.token2idx:
                    self.token2idx[tok] = len(self.idx2token)
                    self.idx2token.append(tok)

    def encode(self, formula: str) -> List[int]:
        tokens = tokenize_formula(formula)
        return [self.token2idx[self.SOS_TOKEN]] + \
               [self.token2idx.get(tok, self.token2idx[self.PAD_TOKEN]) for tok in tokens] + \
               [self.token2idx[self.EOS_TOKEN]]

    def decode(self, token_ids: List[int]) -> str:
        return ' '.join([self.idx2token[idx] for idx in token_ids if idx < len(self.idx2token)])

    def __len__(self):
        return len(self.idx2token)

    def save_to_txt(self, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='latin1') as f:
            for token in self.idx2token:
                f.write(token + '\n')

    @classmethod
    def load_from_txt(cls, path: Path) -> "Vocab":
        vocab = cls()
        with open(path, "r", encoding="latin1") as f:
            for line in f:
                token = line.strip()
                if token not in vocab.token2idx:
                    vocab.token2idx[token] = len(vocab.idx2token)
                    vocab.idx2token.append(token)
        return vocab


def load_caption_formulas(caption_path: Path) -> List[str]:
    formulas = []
    with open(caption_path, "r", encoding="latin1") as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            _, formula = parts
            formulas.append(formula)
    return formulas


formulas = []

vocab = Vocab()
vocab = vocab.load_from_txt("vocab.txt")

vocab.idx2token[0]

'<pad>'

# Encoder

### 변경사항
- 기존 2D attention 대신, 계산 및 메모리 효율, 구현 용이성을 위해 1D attention으로 변경
  - 이때, 어텐션을 위해서는 시퀀스형태로 데이터를 받아와줘야하므로 (B,C,H,W)인 이미지 데이터를 어떻게 (B,T,D) 형태로 변경할지 고민 필요
  - ViT가 아닌 트랜스포머 이전 CNN기반 인코더들에서는 어떻게 이런형태 만들지? Region Proposal하거나 단순히 grid로 쪼개서 사용하나? grid로 쪼갤 경우 위치 정보는 어떻게 부여하지?
  - 일단 코드에는 (B,C,H',W') feature를 H'축에서 평균해서 (B,C,1,W')로 만들고 (B,W',C) 형태로 사용하도록 함, 추후
    - (B, H'W', C) 등 다른방식 고려해볼 수 있을 것

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights


class DenseNetEncoder(nn.Module):

    def __init__(self):
        super().__init__()
        # densenet = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        densenet = densenet121(weights=None)    # 이번 프로젝트는 imageNet 데이터에 비교하면 수식글씨로, imageNet 가중치가 방해될 수도 있어 사전학습 가중치 없이 불러오는게 유리할수도 있음

        # ✅ First conv layer 수정: in_channels=1로 (우리 프로젝트에 맞춰 gpu 메모리 소모 줄이기 위해 그레이스케일로 변환)
        densenet.features.conv0 = nn.Conv2d(
            in_channels=1, out_channels=64,
            kernel_size=7, stride=2, padding=3, bias=False
        )

        # Remove classification head
        features = list(densenet.features.children())

        # Use all layers except final norm+relu+avgpool
        self.backbone = nn.Sequential(*features[:-1])

        # ✅ 가중치 초기화
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.backbone(x)          # (B,1,128,768) -> (B,1024,4,24)
        x = x.mean(dim=2).squeeze()   # (B,1024,4,24) -> (B,1024,1,24) -> (B,1024,24) => 어텐션을 위해 시퀀스형태의 (B,T,D) 꼴로 만들어주기 위한 방법을 H방향을 평균하여 W방향 압축 정보를 시퀀스처럼 사용하는 것으로 선택
        x = x.transpose(1,2)          # (B,1024,24) -> (B,24,1024)
        return x


# Decoder

In [ ]:
import torch
import torch.nn as nn

class Decoder(nn.Module):

    def __init__(self, vocab_size, emb_dim=256, hidden_dim=512, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embed = nn.Embedding(vocab_size, emb_dim)

        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)

        for name, param in self.gru.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, prev_token, hidden):
        embedded = self.embed(prev_token).unsqueeze(1)  # (B, 1, emb_dim)
        output, hidden = self.gru(embedded, hidden)     # (B, 1, H)
        output_logits = self.fc_out(output.squeeze(1))  # (B, vocab_size)
        return output_logits, hidden


# Cross Attention

### 변경사항
- 기존 2D attention 대신, 계산 및 메모리 효율, 구현 용이성을 위해 1D attention으로 변경
- q,k,v projection layer 사용여부, W_o 사용여부는 추후 실험을 통해 선택적으로 사용할 수 있는 옵션으로 쓰면 될듯

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1-D Cross attention
class DualCrossAttention(nn.Module):

    def __init__(self, enc1_dim, enc2_dim, attn_dim=1024, out_dim=1024):
        super().__init__()
        self.q1 = nn.Linear(enc1_dim, attn_dim)
        self.k1 = nn.Linear(enc1_dim, attn_dim)
        self.v1 = nn.Linear(enc1_dim, attn_dim)
        self.o1 = nn.Linear(attn_dim, out_dim)

        self.q2 = nn.Linear(enc2_dim, attn_dim)
        self.k2 = nn.Linear(enc2_dim, attn_dim)
        self.v2 = nn.Linear(enc2_dim, attn_dim)
        self.o2 = nn.Linear(attn_dim, out_dim)

        self.scale = 1.0 / math.sqrt(attn_dim)

    def forward(self, enc1_feats, enc2_feats):

        q1 = self.q1(enc1_feats)            # (B,24,1024)
        k1 = self.k1(enc1_feats)            # (B,24,1024)
        v1 = self.v1(enc1_feats)            # (B,24,1024)

        q2 = self.q2(enc2_feats)            # (B,24,1024)
        k2 = self.k2(enc2_feats)            # (B,24,1024)
        v2 = self.v2(enc2_feats)            # (B,24,1024)

        attn_scores1 = torch.bmm(q1, k2.transpose(1, 2)) * self.scale  # (B,24,24)
        attn_scores2 = torch.bmm(q2, k1.transpose(1, 2)) * self.scale  # (B,24,24)

        attn_weights1 = F.softmax(attn_scores1, dim=-1)  # (B,24,24)
        attn_weights2 = F.softmax(attn_scores2, dim=-1)  # (B,24,24)

        attn_values1 = torch.bmm(attn_weights1, v2)      # (B,24,1024)
        attn_values2 = torch.bmm(attn_weights2, v1)      # (B,24,1024)

        context1 = self.o1(attn_values1)                 # (B,24,1024)
        context2 = self.o2(attn_values2)                 # (B,24,1024)

        context1 = context1.mean(dim=1)                  # (B,1024)
        context2 = context2.mean(dim=1)                  # (B,1024)

        return context1, context2

# DLA

- 이후 추가 모델 개선방안 고민
  - contrastive learning 및 data augmentation (scale augmentation 등)
  - BYOL, DINO 같은 방법 참고해보기? (distilation)
  - DA, DG, SFTA 방법론 참고한 다른 시도
  - multi resolution 기반 실험

In [ ]:
class DLAModel(nn.Module):
    def __init__(self, model_config=None):
        super().__init__()

        # 설정 파라미터 추출
        self.vocab_size = model_config.get("vocab_size", 113)
        self.enc_out_dim = model_config.get("encoder_out_channels", 1024)
        self.dec_emb_dim = model_config.get("decoder_emb_dim", 256)
        self.dec_hid_dim = model_config.get("decoder_hidden_dim", 512)
        self.attn_dim = model_config.get("attention_dim", 1024)
        self.attn_out_dim = model_config.get("attn_out_dim", 1024)

        # 인코더
        self.encoder_pme = DenseNetEncoder()
        self.encoder_hme = DenseNetEncoder()


        # Cross Attention
        self.cross_attention = DualCrossAttention(
            enc1_dim=self.enc_out_dim, enc2_dim=self.enc_out_dim, attn_dim=self.attn_dim, out_dim=self.attn_out_dim
        )

        self.attn_proj1 = nn.Linear(self.enc_out_dim + self.attn_out_dim, self.enc_out_dim)
        self.attn_proj2 = nn.Linear(self.enc_out_dim + self.attn_out_dim, self.enc_out_dim)

        # ✅ 디코더 (PME, HME 동일 구조 → 공유)
        self.decoder = Decoder(self.vocab_size, self.dec_emb_dim, self.dec_hid_dim)

    def forward(self, images_pme, images_hme=None, tgt_pme=None, tgt_hme=None, teacher_forcing_ratio=0.5):

        # images_pme, images_hme: (B,1,128,768)
        # tgt_pme, tgt_hme:       (B,T)

        # PME 인코딩
        feat_pme = self.encoder_pme(images_pme)      # (B,24,1024)

        # HME가 주어진 경우 → paired training
        if images_hme is not None:
            feat_hme = self.encoder_hme(images_hme)   # (B,24,1024)

            # 🔹 양방향 Cross Attention
            context_p, context_h = self.cross_attention(feat_pme, feat_hme)   # (B,24,1024)

            # 🔹 context + hidden → projected hidden
            context_p = self.attn_proj1(torch.cat([feat_pme, context_p], dim=-1))
            context_h = self.attn_proj2(torch.cat([feat_hme, context_h], dim=-1))
        else:
            context_p, context_h = None, None

        # PME 디코딩
        outputs_pme = []

        # <sos>
        prev_token_pme = torch.full((images_pme.size(0), 1), 1, dtype=torch.long, device=images_pme.device)  # (B,1)

        hidden_pme = context_p.copy() if context_p is not None else feat_pme
        for t in range(tgt_pme.size(1)):

            # teacher forcing 안쓰고 <eos>라면 이후부터 <pad>로 예측
            if prev_token_pme == torch.full((images_pme.size(0), 1), 0, dtype=torch.long, device=images_pme.device):
                outputs_pme.append(self.decoder.embed(prev_token_pme).unsqueeze(1))

            logits_pme, hidden_pme = self.decoder(prev_token_pme, hidden_pme)
            outputs_pme.append(logits_pme.unsqueeze(1))
            use_tf = torch.rand(1).item() < teacher_forcing_ratio
            prev_token_pme = tgt_pme[:, t] if use_tf else logits_pme.argmax(dim=-1)

            if use_tf is False and logits_pme.argmax(dim=-1).item() == 2:     # teacher forcing 안쓰고 <eos>라면 이후부터 <pad>로 예측
                prev_token_pme = torch.full((images_pme.size(0), 1), 0, dtype=torch.long, device=images_pme.device)


        # HME 디코딩 (paired인 경우만)
        outputs_hme = []
        if images_hme is not None:
            # <sos>
            prev_token_hme = torch.full((images_hme.size(0), 1), 1, dtype=torch.long, device=images_hme.device)

            hidden_hme = context_h.copy()
            for t in range(tgt_hme.size(1)):

                # teacher forcing 안쓰고 <eos>라면 이후부터 <pad>로 예측
                if prev_token_hme == torch.full((images_hme.size(0), 1), 0, dtype=torch.long, device=images_hme.device):
                    outputs_hme.append(self.decoder.embed(prev_token_hme).unsqueeze(1))

                logits_hme, hidden_hme = self.decoder(prev_token_hme, hidden_hme)
                outputs_hme.append(logits_hme.unsqueeze(1))
                use_tf = torch.rand(1).item() < teacher_forcing_ratio
                prev_token_hme = tgt_hme[:, t] if use_tf else logits_hme.argmax(dim=-1)

                if use_tf is False and logits_hme.argmax(dim=-1).item() == 2:     # teacher forcing 안쓰고 <eos>라면 이후부터 <pad>로 예측
                    prev_token_hme = torch.full((images_hme.size(0), 1), 0, dtype=torch.long, device=images_hme.device)

        logits_pme = torch.cat(outputs_pme, dim=1)
        logits_hme = torch.cat(outputs_hme, dim=1) if outputs_hme else None

        return logits_pme, logits_hme

# Custom Loss (여기서부터 작업필요)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DualLoss(nn.Module):
    """
    Dual loss function for paired dual loss attention model.

    Loss = LD(Xh) + LD(Xp) + LD(X̄) + λ * Lmatch(Xh, Xp)
    """
    def __init__(self, match_weight=0.2, ignore_index=0):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.mse_loss = nn.MSELoss()
        self.match_weight = match_weight

    def forward(self,
                logits_h, targets_h,        # (B, T, V), (B, T)
                logits_p, targets_p,        # (B, T, V), (B, T)
                logits_up=None, targets_up=None,  # (B, T, V), (B, T)
                context_h=None, context_p=None    # (B, T, C), (B, T, C)
               ):
        """
        logits_*: unnormalized decoder outputs (B, T, V)
        targets_*: target indices (B, T)
        context_*: attention context vectors (B, T, C)
        """
        loss_total = 0.0

        # Decoder Loss - Handwritten
        if logits_h is not None and targets_h is not None:
            B, T, V = logits_h.size()
            targets_h = targets_h[:, 1:]
            loss_h = self.ce_loss(logits_h.reshape(-1, V), targets_h.reshape(-1))
            loss_total += loss_h

        # Decoder Loss - Paired Printed
        if logits_p is not None and targets_p is not None:
            B, T, V = logits_p.size()
            targets_p = targets_p[:, 1:]
            loss_p = self.ce_loss(logits_p.reshape(-1, V), targets_p.reshape(-1))
            loss_total += loss_p

        # Decoder Loss - Unpaired PME
        if logits_up is not None and targets_up is not None:
            B, T, V = logits_up.size()
            targets_up = targets_up[:, 1:]
            loss_up = self.ce_loss(logits_up.reshape(-1, V), targets_up.reshape(-1))
            loss_total += loss_up

        else:
            loss_up = torch.tensor(0.0)

        # Context Matching Loss
        if context_h is not None and context_p is not None:
            assert context_h.shape == context_p.shape
            match_loss = self.mse_loss(context_h, context_p)
            loss_total += self.match_weight * match_loss
        else:
            match_loss = torch.tensor(0.0)

        return loss_total, {
            "loss_h": loss_h.item(),
            "loss_p": loss_p.item(),
            "loss_up": loss_up.item(),
            "match_loss": match_loss.item(),
            "total": loss_total.item()
        }


# Utils

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed: int = 42):
    """재현 가능한 실험을 위한 시드 고정."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"🌱 Seed set to {seed}")

In [ ]:
import os
import json

def save_log(log_dict, save_path="log.json"):
    """학습 로그를 JSON으로 저장."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(log_dict, f, indent=2, ensure_ascii=False)
    print(f"📄 로그 저장 완료: {save_path}")


In [ ]:
import torch
import numpy as np
from difflib import SequenceMatcher


def token_accuracy(preds, targets, pad_token=0):
    """
    Token-level accuracy (ignoring pad_token).
    """
    mask = targets != pad_token
    correct = (preds == targets) & mask
    accuracy = correct.sum().item() / mask.sum().item()
    return accuracy


def exprate_k(preds, targets, k):
    """
    Expression rate-k: 예측 수식과 정답 수식이 k개 이하의 토큰 차이만 있을 때 정답으로 간주.
    길이 차이도 diff에 포함하여 평가.
    """
    assert len(preds) == len(targets)
    correct = 0
    for p, t in zip(preds, targets):
        diff = abs(len(p) - len(t))
        for a, b in zip(p, t):
            if a != b:
                diff += 1
        if diff <= k:
            correct += 1
    return correct / len(preds)


def cer(preds, targets):
    """
    Character Error Rate (token-level edit distance / length of target).
    평균 CER을 전체 샘플에 대해 계산.
    """
    total_distance = 0
    total_length = 0
    for p, t in zip(preds, targets):
        total_distance += levenshtein_distance(p, t)
        total_length += len(t)
    return total_distance / total_length if total_length > 0 else 0


def levenshtein_distance(seq1, seq2):
    """
    기본적인 편집 거리 계산 함수 (DP 기반).
    """
    n, m = len(seq1), len(seq2)
    dp = np.zeros((n + 1, m + 1), dtype=np.int32)

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )
    return dp[n][m]

In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_loss_curve(loss_list, save_path=None):
    """Loss 곡선 시각화."""
    plt.figure(figsize=(8, 4))
    plt.plot(loss_list, marker='o', label="Train Loss")
    plt.title("Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        print(f"📊 Loss curve saved to {save_path}")
    else:
        plt.show()

def visualize_prediction(image, pred_str, target_str, figsize=(10, 3)):
    """수식 이미지와 예측/정답 문자열 시각화"""
    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=figsize)
    plt.imshow(image.squeeze(), cmap='gray')
    plt.title(f"Pred: {pred_str}\nTarget: {target_str}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def plot_learning_curve(losses, title="Training Loss"):
    plt.plot(losses, label="loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid()
    plt.legend()
    plt.show()

# Train

In [ ]:
experiment_name: "dla_baseline"

# ✅ 데이터 경로
data:
  vocab: "data/vocab/crohme_vocab.txt"
  paired:
    hme_img: "data/preprocessed/train/paired/crohme/hme_preprocessed"
    pme_img: "data/preprocessed/train/paired/crohme/pme_preprocessed"
    caption: "data/preprocessed/train/paired/crohme/caption.txt"
  unpaired:
    pme_img: "data/preprocessed/train/paired/im2latex/pme_cropped"
    caption: "data/preprocessed/train/paired/im2latex/caption.txt"

# ✅ 모델 하이퍼파라미터
model:
  vocab_size: 113
  encoder_out_channels: 1024
  decoder_emb_dim: 256
  decoder_hidden_dim: 512
  attention_dim: 1024
  attn_out_dim: 1024

# ✅ 학습 설정
training:
  batch_size: 1
  epochs: 100
  learning_rate: 0.1
  match_weight: 0.2
  optimizer: "adadelta"
  grad_clip: 5.0
  early_stop_patience: 5
  ignore_idx: 0

  scheduler:
    use: true
    type: "StepLR"
    step_size: 10
    gamma: 0.5

# ✅ 이미지 전처리 설정
transforms:
  paired_hme:
    target_size: [481, 2116] # width, height 순서가 아니라 height, width 순서임에 유의
    mean: [0.0770]
    std: [0.2633]

  paired_pme:
    target_size: [125, 872]
    mean: [0.8126]
    std: [0.3817]

  unpaired:
    target_size: [209, 1362]
    mean: [0.9228]
    std: [0.2654]


# ✅ 테스트 설정
testing:
  batch_size: 1
  max_len: 150

# ✅ 기타 설정
misc:
  seed: 42
  device: "mps"   # "cuda" / "cpu"
  checkpoint_path: "runs/dla_baseline_bs1_lr0.1_match0.2_schedStepLR3_0.5_seed42/batch_logs/best_model.pth"
  batch_log_path: "runs/dla_baseline_bs1_lr0.1_match0.2_schedStepLR3_0.5_seed42/batch_logs/best_model.pth"

In [ ]:
import torch
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm
import yaml
import argparse
from datetime import datetime

# ✅ config 불러오기
def load_config(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, default="config.yaml")
args = parser.parse_args()
config = load_config(args.config)

# ✅ 하이퍼파라미터 기반 실험 이름 설정
EXP_NAME = (
    f"{config['experiment_name']}"
    f"_bs{config['training']['batch_size']}"
    f"_lr{config['training']['learning_rate']}"
    f"_match{config['training']['match_weight']}"
)

if config["training"]["scheduler"]["use"]:
    sched = config["training"]["scheduler"]
    EXP_NAME += f"_sched{sched['type']}{sched['step_size']}_{sched['gamma']}"

base_dir = Path(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../")))

EXP_NAME += f"_seed{config['misc']['seed']}"
SAVE_ROOT = base_dir / Path("runs") / EXP_NAME
SAVE_DIR = SAVE_ROOT / "checkpoints"
BATCH_LOG_DIR = SAVE_ROOT / "batch_logs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
BATCH_LOG_DIR.mkdir(parents=True, exist_ok=True)

# ✅ 하이퍼파라미터 세팅
BATCH_SIZE = config["training"]["batch_size"]
EPOCHS = config["training"]["epochs"]
LEARNING_RATE = config["training"]["learning_rate"]
MATCH_WEIGHT = config["training"]["match_weight"]
IGNORE_IDX = config["training"]["ignore_idx"]
PATIENCE_THRESHOLD = config["training"]["early_stop_patience"]
GRAD_CLIP = config["training"]["grad_clip"]

# ✅ device 설정
DEVICE = torch.device(config["misc"]["device"] if torch.backends.mps.is_available() or torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {DEVICE}")

# ✅ 시드 고정
# set_seed(config["misc"]["seed"])

# ✅ vocab

vocab = Vocab.load_from_txt(base_dir / Path(config["data"]["vocab"]))
vocab_size = len(vocab)

# ✅ transform
hme_paired_transform = get_formula_transform("paired_hme", config["transforms"])
pme_paired_transform = get_formula_transform("paired_pme", config["transforms"])
unpaired_transform = get_formula_transform("unpaired", config["transforms"])

# ✅ Dataset
paired_caption_path = base_dir / Path(config["data"]["paired"]["caption"])
paired_dataset = PairedFormulaDataset(
    hme_dir=base_dir / Path(config["data"]["paired"]["hme_img"]),
    pme_dir=base_dir / Path(config["data"]["paired"]["pme_img"]),
    caption_path=paired_caption_path,
    transform_hme=hme_paired_transform,
    transform_pme=pme_paired_transform,
    vocab=vocab
)
unpaired_dataset = FormulaDataset(
    image_dir=base_dir / Path(config["data"]["unpaired"]["pme_img"]),
    caption_path=base_dir / Path(config["data"]["unpaired"]["caption"]),
    transform=unpaired_transform,
    vocab=vocab
)

# ✅ Dataloader
collate_fn_1 = lambda b: formula_collate_fn(b, pad_idx=IGNORE_IDX)
collate_fn_2 = lambda b: paired_collate_fn(b, pad_idx=IGNORE_IDX)
unpaired_loader = DataLoader(unpaired_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn_1)
paired_loader = DataLoader(paired_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn_2)


# ✅ 모델, 손실함수, 옵티마이저
model = DLAModel(vocab_size=vocab_size, model_config=config["model"]).to(DEVICE)
criterion = DualLoss(match_weight=MATCH_WEIGHT, ignore_index=IGNORE_IDX)

# 🔧 optimizer 선택
optimizer_name = config["training"].get("optimizer", "adadelta").lower()
params = model.parameters()
if optimizer_name == "adam":
    optimizer = torch.optim.Adam(params, lr=LEARNING_RATE)
elif optimizer_name == "sgd":
    optimizer = torch.optim.SGD(params, lr=LEARNING_RATE, momentum=0.9)
elif optimizer_name == "adadelta":
    optimizer = torch.optim.Adadelta(params, lr=LEARNING_RATE)
else:
    raise ValueError(f"❌ Unknown optimizer: {optimizer_name}")

# ✅ learning rate scheduler
scheduler = None
sched_cfg = config["training"]["scheduler"]
if sched_cfg["use"]:
    if sched_cfg["type"] == "StepLR":
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=sched_cfg["step_size"], gamma=sched_cfg["gamma"])
    else:
        raise NotImplementedError(f"{sched_cfg['type']} scheduler is not supported.")

# ✅ 학습 루프
best_loss = float("inf")
patience = 0
loss_history, log_dict = [], {"train": []}
unpaired_iter = iter(unpaired_loader)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_acc, num_batches = 0.0, 0.0, 0

    loop = tqdm(paired_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)
    for batch_pair in loop:
        try:
            batch_up = next(unpaired_iter)
        except StopIteration:
            unpaired_iter = iter(unpaired_loader)
            batch_up = next(unpaired_iter)

        img_pme, img_hme = batch_pair["img_pme"].to(DEVICE), batch_pair["img_hme"].to(DEVICE)
        tgt = batch_pair["formula"].to(DEVICE)
        img_up, tgt_up = batch_up["image"].to(DEVICE), batch_up["formula"].to(DEVICE)

        logits_p, logits_h, context_p, context_h = model(img_pme, img_hme, tgt, tgt)
        logits_up, _, _, _ = model(img_up, None, tgt_up, None)

        loss, loss_dict = criterion(logits_h, tgt, logits_p, tgt, logits_up, tgt_up, context_h, context_p)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        optimizer.step()

        preds = logits_h.argmax(dim=-1)
        acc = token_accuracy(preds, tgt[:, 1:])

        total_loss += loss.item()
        total_acc += acc
        num_batches += 1

        torch.save({
            "epoch": epoch,
            "batch_idx": num_batches,
            "loss_dict": loss_dict,
            "loss": loss.item(),
            "token_accuracy": acc,
            "model_state_dict": model.state_dict()
        }, BATCH_LOG_DIR / f"epoch{epoch:03d}_batch{num_batches:03d}.pt"),

        loop.set_postfix(loss=loss.item(), acc=acc)

    if num_batches == 0:
        print("⚠️ No batch processed.")
        continue

    avg_loss = total_loss / num_batches
    avg_acc = total_acc / num_batches
    print(f"\n📊 [Epoch {epoch}] Loss: {avg_loss:.4f}, Token Acc: {avg_acc:.4f}")

    log_dict["train"].append({
        "epoch": epoch, "loss": avg_loss, "token_accuracy": avg_acc
    })
    loss_history.append(avg_loss)

    if scheduler:
        scheduler.step()

    if epoch == 1 or avg_loss < best_loss:
        best_loss = avg_loss
        patience = 0
        torch.save({
            "model": model.state_dict()
        }, SAVE_DIR / "best_model.pth")
        print("🧠 Best model saved!")
    else:
        patience += 1
        if patience >= PATIENCE_THRESHOLD:
            print(f"🛑 Early stopping at epoch {epoch}")
            break

# ✅ 최종 저장
save_log(log_dict, save_path=SAVE_ROOT / "train_log.json")
plot_loss_curve(loss_history, save_path=SAVE_ROOT / "loss_curve.png")
torch.save({
    "model": model.state_dict()
}, SAVE_DIR / "final_model.pth")
print("✅ 최종 모델 저장 완료!")

# Test

In [ ]:
import json
import yaml
import argparse
from pathlib import Path
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader

from dla import DLAModel
from data.dataset import FormulaDataset
from data.utils.vocab import Vocab
from utils.metrics import exprate_k, cer
from utils.decode import decode_sequence
from utils.seed import set_seed
from utils.data_utils import get_formula_transform, formula_collate_fn

# ✅ config.yaml 불러오기
def load_config(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, default="config.yaml")
args = parser.parse_args()
config = load_config(args.config)

base_dir = Path(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../")))

# ✅ 기본 설정값 로드
set_seed(config["misc"]["seed"])
IGNORE_IDX = config["training"]["ignore_idx"]
BATCH_SIZE = config["testing"].get("batch_size", 1)
MAX_LEN = config["testing"].get("max_len", 150)
TEST_YEARS = ["2014", "2016", "2019"]
CHECKPOINT_PATH = base_dir / config["misc"]["checkpoint_path"]
VOCAB_PATH = config["data"]["vocab"]
DEVICE = torch.device(config["misc"]["device"] if torch.backends.mps.is_available() or torch.cuda.is_available() else "cpu")

# ✅ 결과 디렉토리
os.makedirs("preds", exist_ok=True)

# ✅ vocab 및 모델 로드
vocab = Vocab.load_from_txt(base_dir / Path(VOCAB_PATH))
SOS_ID = vocab.token2idx["<sos>"]
EOS_ID = vocab.token2idx["<eos>"]

model_config = config["model"]
unpaired_model = DLAModel(vocab_size=len(vocab), model_config=model_config).to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
unpaired_model.load_state_dict(checkpoint["model"])
unpaired_model.eval()

# ✅ 평가 함수 정의
def evaluate(img_dir, caption_path, img_ext, mode="hme"):
    assert mode in ["hme", "pme"], f"❌ Invalid mode: {mode}"

    transform = get_formula_transform("paired_hme" if mode == "hme" else "paired_pme", config["transforms"])
    dataset = FormulaDataset(
        image_dir=img_dir,
        caption_path=caption_path,
        transform=transform,
        vocab=vocab
    )
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: formula_collate_fn(b, pad_idx=IGNORE_IDX)
    )

    preds, targets = [], []
    debug_lines = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"{mode.upper()} Eval"):
            imgs = batch["image"].to(DEVICE)
            tgt_ids = batch["formula"]

            pred_ids_batch = unpaired_model.predict(imgs, max_len=MAX_LEN, sos_idx=SOS_ID, eos_idx=EOS_ID)
            pred_tokens_batch = decode_sequence(pred_ids_batch, vocab)
            target_tokens_batch = decode_sequence(tgt_ids, vocab)

            preds.extend(pred_tokens_batch)
            targets.extend(target_tokens_batch)

            for pred, target in zip(pred_tokens_batch, target_tokens_batch):
                debug_lines.append(f"[GT]   {' '.join(target)}")
                debug_lines.append(f"[PRD]  {' '.join(pred)}")
                debug_lines.append("---")

    year = Path(img_dir).parts[-2]
    debug_path = base_dir / f"preds/pred_target_pairs_{mode}_{year}.txt"
    with open(debug_path, "w", encoding="latin1") as f:
        f.write("\n".join(debug_lines))
    print(f"📄 {mode.upper()} {year} 결과 저장됨: {debug_path}")

    return preds, targets

# ✅ 결과 출력 함수
def print_result_section(title: str, result_dict: dict):
    print(f"\n📊 {title}")
    for key, val in result_dict.items():
        print(f"  {key}: {val}")

# ✅ 전체 평가
result_log = {"CROHME": {"hme": {}, "pme": {}}, "IM2LATEX": {"pme":{}}}

# hme 평가
for year in TEST_YEARS:
    img_dir = base_dir / f"data/preprocessed/test/hme/crohme/{year}/hme_img"
    caption_path = base_dir / f"data/preprocessed/test/hme/crohme/{year}/caption.txt"
    preds, targets = evaluate(img_dir, caption_path, img_ext="bmp", mode="hme")

    result_log["CROHME"]["hme"][year] = {
        f"exprate_{k}": round(exprate_k(preds, targets, k), 4) for k in range(4)
    }
    result_log["CROHME"]["hme"][year]["cer"] = round(cer(preds, targets), 4)

# pme 평가
for year in TEST_YEARS:
    img_dir = base_dir / f"data/preprocessed/test/pme/crohme/{year}/pme_img"
    caption_path = base_dir / f"data/preprocessed/test/pme/crohme/{year}/caption.txt"
    preds, targets = evaluate(img_dir, caption_path, img_ext="png", mode="pme")

    result_log["CROHME"]["pme"][year] = {
        f"exprate_{k}": round(exprate_k(preds, targets, k), 4) for k in range(4)
    }
    result_log["CROHME"]["pme"][year]["cer"] = round(cer(preds, targets), 4)

# im2latex 평가
img_dir = base_dir / f"data/preprocessed/test/pme/im2latex/img"
caption_path = base_dir / f"data/preprocessed/test/pme/im2latex/caption.txt"
preds, targets = evaluate(img_dir, caption_path, img_ext="png", mode="pme")

result_log["IM2LATEX"]["pme"] = {
    f"exprate_{k}": round(exprate_k(preds, targets, k), 4) for k in range(4)
}
result_log["IM2LATEX"]["pme"]["cer"] = round(cer(preds, targets), 4)

# ✅ 출력
for year in TEST_YEARS:
    print_result_section(f"CROHME-HME {year}", result_log["CROHME"]["hme"][year])
    print_result_section(f"CROHME-PME {year}", result_log["CROHME"]["pme"][year])

print_result_section("IM2LATEX-PME", result_log["IM2LATEX"]["pme"])

# ✅ 전체 결과 저장
with open("test_results.json", "w") as f:
    json.dump(result_log, f, indent=4)

print("\n✅ 평가 완료! 결과는 test_results.json에 저장됨")
